# AKE Lab 4 · Setup-Check (einmalig, mit Internet)

Lab 4 passt **SmolLM2-135M-Instruct** an. Konfiguration und Tokenizer liegen im Kursrepository unter
`data/models/SmolLM2-135M-Instruct/`. Die **Gewichte** (`model.safetensors`, ≈ 270 MB) sind zu groß für GitHub (Limit 100 MB
pro Datei) und werden **einmal** mit diesem Notebook vom Hugging-Face-Hub geladen – am besten zu Hause oder im Hochschul-WLAN
**vor** dem Lab. Im Lab selbst gibt es keinen Hub-Zugriff.

**Ablauf:** Zelle für Zelle mit *Shift + Enter*. Erwartung: alle Prüfungen ✅, Dauer je nach Leitung 1–5 Min. Zelle 4 rechnet aus
drei Trainingsschritten hoch, wie lange das Training in Aufgabe 4 auf Ihrem Rechner dauert.

**Kein Internet oder Hub gesperrt?** Die Datei `model.safetensors` vom Hochschul-Share (Ordner *Lab04 / models /
SmolLM2-135M-Instruct*) nach `data/models/SmolLM2-135M-Instruct/` kopieren und Zelle 2 erneut ausführen.
Neue Pakete werden nicht installiert; `peft` ist seit dem ersten `uv sync` vorhanden.

In [ ]:
# 1 · Umgebung und Pfade
import os, sys, hashlib, importlib, time
from pathlib import Path

REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Kein Kursrepository gefunden: Jupyter im Ordner des Repositorys starten und lab04/ oeffnen.")
DATA  = REPO_ROOT / "data"
ZIEL  = DATA / "models" / "SmolLM2-135M-Instruct"
DATEI = ZIEL / "model.safetensors"
REPO_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"
SHA256_OK = "5af571cbf074e6d21a03528d2330792e532ca608f24ac70a143f6b369968ab8c"

def check(name, ok, hint=""):
    print(f"✅ {name}" if ok else f"❌ {name} – Hinweis: {hint}")
    return ok

print("Python", sys.version.split()[0], "| REPO_ROOT:", REPO_ROOT)
for paket in ["torch", "transformers", "peft", "huggingface_hub", "safetensors", "pandas", "matplotlib"]:
    try:
        m = importlib.import_module(paket)
        check(f"{paket} {getattr(m, '__version__', '')}", True)
    except ImportError:
        check(paket, False, "fehlt -> im Repository `uv sync` ausfuehren")
check("SmolLM2-Konfiguration und -Tokenizer im Repository",
      all((ZIEL / f).exists() for f in ["config.json", "tokenizer.json", "tokenizer_config.json"]), "git pull im Repository")
check("Daten data/stoermeldungen/ im Repository",
      all((DATA / "stoermeldungen" / f"{k}.jsonl").exists() for k in ("train", "val", "eval", "regression")), "git pull")


In [ ]:
# 2 · Gewichte laden (nur, wenn sie noch fehlen) und pruefen
if DATEI.exists():
    print(f"{DATEI.relative_to(REPO_ROOT)} ist bereits vorhanden ({DATEI.stat().st_size / 1e6:.0f} MB) – kein Download.")
else:
    from huggingface_hub import hf_hub_download
    print("Lade model.safetensors (≈ 270 MB) von huggingface.co/HuggingFaceTB/SmolLM2-135M-Instruct ...")
    hf_hub_download(REPO_ID, "model.safetensors", local_dir=ZIEL)

h = hashlib.sha256()
with open(DATEI, "rb") as f:
    for block in iter(lambda: f.read(1 << 20), b""):
        h.update(block)
check("Pruefsumme stimmt (gleiche Gewichte wie im Referenzlauf)", h.hexdigest() == SHA256_OK,
      "Datei unvollstaendig oder andere Version: loeschen und Zelle erneut ausfuehren")


In [ ]:
# 3 · Ladetest offline: genau so laedt das Lab das Modell
os.environ["HF_HUB_OFFLINE"] = "1"; os.environ["TRANSFORMERS_OFFLINE"] = "1"
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.utils import logging
logging.set_verbosity_error()
from peft import LoraConfig, get_peft_model
torch.set_num_threads(min(4, os.cpu_count()))

tok = AutoTokenizer.from_pretrained(ZIEL)
model = AutoModelForCausalLM.from_pretrained(ZIEL, torch_dtype=torch.float32).eval()
n_param = sum(p.numel() for p in model.parameters())
check(f"SmolLM2 laedt: {n_param / 1e6:.1f} Mio. Parameter", abs(n_param - 134_515_008) < 1e6)
msgs = [{"role": "user", "content": "Was ist 2 + 3?"}]
enc = tok(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True), return_tensors="pt", add_special_tokens=False)
with torch.inference_mode():
    out = model.generate(**enc, do_sample=False, max_new_tokens=20, pad_token_id=tok.pad_token_id)
print("Testantwort:", repr(tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)))


In [ ]:
# 4 · Rechenzeit hochrechnen: 3 LoRA-Trainingsschritte (Batch 4, gepaddete Laenge wie im Lab) -> 150 Schritte
torch.manual_seed(0)
lm = get_peft_model(model, LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, target_modules=["q_proj", "v_proj"],
                                      task_type="CAUSAL_LM")).train()
opt = torch.optim.AdamW([p for p in lm.parameters() if p.requires_grad], lr=2e-4)
ids = torch.randint(100, 40000, (4, 195)); lab = ids.clone()
t = []
for i in range(4):
    t0 = time.perf_counter()
    loss = lm(input_ids=ids, labels=lab).loss; opt.zero_grad(); loss.backward(); opt.step()
    t.append(time.perf_counter() - t0)
s = sum(t[1:]) / 3                                     # erster Schritt = Aufwaermen
print(f"{s:.2f} s je Schritt (Threads: {torch.get_num_threads()}) -> 150 Schritte ≈ {150 * s / 60:.1f} Min.")
check("Training in Aufgabe 4 dauert hochgerechnet hoechstens 20 Min.", 150 * s / 60 <= 20,
      "Rechner langsam: in Aufgabe 4 NUTZE_REFERENZ_ADAPTER = True setzen (mitgelieferter Adapter)")
print("\nFertig. Lab 4 kann offline starten: lab04_starter.ipynb als lab04_<nachname>.ipynb kopieren.")
